In [3]:
# ================================
# INSTALL DEPENDENCIES
# ================================
!pip install -q gradio transformers torch black autopep8 reportlab requests

# ================================
# IMPORT LIBRARIES
# ================================
import gradio as gr
import torch
import ast
import re
import black
import autopep8
from transformers import pipeline
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import datetime

# ================================
# CONFIGURATION
# ================================
class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 256
    DEFAULT_TOP_P = 0.95


# ================================
# LOAD MODEL
# ================================
print("Loading IBM Granite model...")

device = 0 if torch.cuda.is_available() else -1

model_pipe = pipeline(
    "text-generation",
    model=ChatbotConfig.MODEL_NAME,
    device=device
)

print("Model Loaded Successfully")

# ================================
# CONVERSATION HISTORY
# ================================
conversation_history = []


# ================================
# PYTHON CODE DETECTION
# ================================
def detect_python_code(text):

    patterns = [
        r'def\s+\w+\s*\(',
        r'class\s+\w+',
        r'import\s+\w+',
        r'from\s+\w+\s+import',
        r'if\s+.*:',
        r'for\s+.*\s+in\s+',
        r'while\s+.*:'
    ]

    for pattern in patterns:
        if re.search(pattern, text):
            return True

    return False


# ================================
# SYNTAX CHECK
# ================================
def check_syntax(code):

    try:
        ast.parse(code)
        return True, "No syntax errors"

    except SyntaxError as e:
        return False, f"Syntax Error (Line {e.lineno}): {e.msg}"


# ================================
# FIX INDENTATION
# ================================
def fix_code_indentation(code):

    code = code.replace("\t", "    ")
    lines = code.split("\n")

    cleaned = []

    for line in lines:
        cleaned.append(line.rstrip())

    return "\n".join(cleaned)


# ================================
# FIX BRACKETS
# ================================
def fix_code_brackets(code):

    brackets = {
        "(": ")",
        "[": "]",
        "{": "}"
    }

    for open_b, close_b in brackets.items():

        if code.count(open_b) > code.count(close_b):
            code += close_b

    return code


# ================================
# FORMAT CODE
# ================================
def format_code(code):

    try:
        formatted = black.format_str(code, mode=black.FileMode())
        return formatted

    except:
        formatted = autopep8.fix_code(code)
        return formatted


# ================================
# FULL CODE FIX ENGINE
# ================================
def fix_python_code(code):

    code = fix_code_indentation(code)
    code = fix_code_brackets(code)
    code = format_code(code)

    return code


# ================================
# AI RESPONSE GENERATION
# ================================
def generate_response(message, temperature, max_tokens, top_p):

    conversation_history.append(
        {"role": "user", "content": message}
    )

    response = model_pipe(
        message,
        max_new_tokens=int(max_tokens),
        temperature=temperature,
        top_p=top_p
    )

    reply = response[0]["generated_text"]

    conversation_history.append(
        {"role": "assistant", "content": reply}
    )

    return reply


# ================================
# CHAT FUNCTION
# ================================
def chat(message, history, temperature, max_tokens, top_p):

    if detect_python_code(message):

        valid, error = check_syntax(message)

        if valid:

            fixed = format_code(message)

            return f"✅ Code is correct\n\nFormatted Code:\n{fixed}"

        else:

            fixed = fix_python_code(message)

            return f"⚠️ {error}\n\n🔧 Fixed Code:\n{fixed}"

    else:

        reply = generate_response(message, temperature, max_tokens, top_p)

        return reply


# ================================
# TXT EXPORT
# ================================
def download_txt():

    text = ""

    for msg in conversation_history:
        role = msg["role"].upper()
        text += f"{role}: {msg['content']}\n\n"

    filename = "chat_history.txt"

    with open(filename, "w") as f:
        f.write(text)

    return filename


# ================================
# PDF EXPORT
# ================================
def download_pdf():

    filename = "chat_history.pdf"

    c = canvas.Canvas(filename, pagesize=letter)

    y = 750

    for msg in conversation_history:

        text = f"{msg['role'].upper()}: {msg['content']}"

        for line in text.split("\n"):

            c.drawString(40, y, line)
            y -= 15

            if y < 40:
                c.showPage()
                y = 750

    c.save()

    return filename


# ================================
# CLEAR CHAT
# ================================
def clear_chat():

    global conversation_history
    conversation_history = []

    return "Chat Cleared"


# ================================
# GRADIO INTERFACE
# ================================
with gr.Blocks(title="Syntax Surgeon") as app:

    gr.Markdown("# 🩺 Syntax Surgeon")
    gr.Markdown("AI Chatbot + Python Code Fixer using IBM Granite")

    chatbot = gr.ChatInterface(
        fn=chat,
        additional_inputs=[
            gr.Slider(0.1,1,value=0.7,label="Temperature"),
            gr.Slider(50,1024,value=256,label="Max Tokens"),
            gr.Slider(0.1,1,value=0.95,label="Top P")
        ]
    )

    with gr.Row():

        txt_btn = gr.Button("Download TXT")
        pdf_btn = gr.Button("Download PDF")
        clear_btn = gr.Button("Clear Chat")

    file_output = gr.File()

    txt_btn.click(download_txt, outputs=file_output)
    pdf_btn.click(download_pdf, outputs=file_output)
    clear_btn.click(clear_chat)

app.launch(share=True)

Loading IBM Granite model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/787 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model Loaded Successfully


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://759c6bde5711cde0b1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
